# Data 140: Intro to Machine Learning (Interactive)

## How to use this notebook

**Important:** Students should interact with **widgets** (checkboxes/sliders) instead of editing code.

### Step-by-step workflow
1. Run cells **top to bottom** once.
2. In each interactive section, change widget values.
3. Record results in the reflection prompts.
4. Do **not** modify code unless your instructor asks you to.

---

## Learning goals
- Explain the difference between **features (X)** and **label (y)**.
- Observe how feature choice changes model quality.
- Compare regression and classification workflows.


In [ ]:
# Setup (run once)
from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import mean_squared_error, accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from IPython.display import display, Markdown

try:
    import ipywidgets as widgets
except ImportError as e:
    raise ImportError(
        "This notebook requires ipywidgets. Install it with: pip install ipywidgets"
    ) from e


# Part 1) Supervised Learning — Regression

We will predict California median house value (`MedHouseVal`).

## Dataset features
- `MedInc`: median income
- `HouseAge`: median house age
- `AveRooms`: average rooms
- `AveBedrms`: average bedrooms
- `Population`: neighborhood population
- `AveOccup`: average occupancy
- `Latitude`, `Longitude`: location coordinates

### Your task
Use the widgets to test different feature combinations and test sizes.


In [ ]:
# Load and preview California housing data
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

display(df.head())
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")


## Interactive Regression Lab

### Parameter guide (what each control means)
- **Features**: input columns used to predict `MedHouseVal`.
- **Test size**: fraction of rows held out for testing (e.g., `0.30` means 30% test, 70% train).
- **Random seed**: controls the train/test split; changing it gives a different split.
- **Fit intercept**: if checked, the model learns a baseline constant term.
- **Feature for line plot**: x-axis feature used for the live best-fit line preview.

### Exactly what to do
1. In **Features**, select 1+ features.
2. Move **Test size** and **Random seed** to see split sensitivity.
3. Toggle **Fit intercept** and compare MSE.
4. Inspect the two plots (train and test best-fit line view).
5. Repeat at least 4 trials and record patterns.

### Interpretation tips
- Lower **MSE** is better.
- If train MSE is much lower than test MSE, the model may not generalize as well.
- The line plots are easiest to interpret when the x-feature has a strong relationship with `MedHouseVal`.


In [ ]:
all_regression_features = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms",
    "Population", "AveOccup", "Latitude", "Longitude"
]

reg_feature_widget = widgets.SelectMultiple(
    options=all_regression_features,
    value=("MedInc", "HouseAge"),
    description="Features",
    rows=8,
    layout=widgets.Layout(width="420px"),
)

reg_plot_feature_widget = widgets.Dropdown(
    options=all_regression_features,
    value="MedInc",
    description="Plot x",
)

reg_test_size_widget = widgets.FloatSlider(
    value=0.2,
    min=0.1,
    max=0.6,
    step=0.05,
    description="Test size",
    continuous_update=False,
)

reg_seed_widget = widgets.IntSlider(
    value=42,
    min=0,
    max=200,
    step=1,
    description="Random seed",
    continuous_update=False,
)

reg_intercept_widget = widgets.Checkbox(
    value=True,
    description="Fit intercept",
)


def run_regression(selected_features, test_size, random_seed, fit_intercept, plot_feature):
    if len(selected_features) == 0:
        display(Markdown("❗Please select at least one feature."))
        return

    X = df[list(selected_features)]
    y = df["MedHouseVal"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_seed
    )

    model = LinearRegression(fit_intercept=fit_intercept)
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)

    display(Markdown(f"**Selected features:** `{list(selected_features)}`"))
    display(Markdown(f"**Test size:** `{test_size:.2f}` | **Random seed:** `{random_seed}` | **Fit intercept:** `{fit_intercept}`"))
    display(Markdown(f"**Train MSE:** `{train_mse:.4f}` | **Test MSE:** `{test_mse:.4f}` (lower is better)"))

    if plot_feature not in df.columns:
        display(Markdown("⚠️ Plot feature not available."))
        return

    if plot_feature not in selected_features:
        display(Markdown(f"ℹ️ `{plot_feature}` is used only for visualization (model still uses selected features)."))

    # 1D visualization model for clear best-fit line on chosen x-axis feature
    X_line = df[[plot_feature]]
    Xl_train, Xl_test, yl_train, yl_test = train_test_split(
        X_line, y, test_size=test_size, random_state=random_seed
    )
    line_model = LinearRegression(fit_intercept=fit_intercept)
    line_model.fit(Xl_train, yl_train)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

    for ax, X_part, y_part, title in [
        (axes[0], Xl_train, yl_train, "Train set best-fit line"),
        (axes[1], Xl_test, yl_test, "Test set best-fit line"),
    ]:
        x_vals = X_part[plot_feature].to_numpy()
        y_vals = y_part.to_numpy()
        order = np.argsort(x_vals)
        x_sorted = x_vals[order]
        y_line = line_model.predict(pd.DataFrame({plot_feature: x_sorted}))

        ax.scatter(x_vals, y_vals, alpha=0.35, s=14)
        ax.plot(x_sorted, y_line, color="black", linewidth=2)
        ax.set_title(title)
        ax.set_xlabel(plot_feature)
        ax.grid(alpha=0.2)

    axes[0].set_ylabel("MedHouseVal")
    plt.tight_layout()
    plt.show()

reg_output = widgets.interactive_output(
    run_regression,
    {
        "selected_features": reg_feature_widget,
        "test_size": reg_test_size_widget,
        "random_seed": reg_seed_widget,
        "fit_intercept": reg_intercept_widget,
        "plot_feature": reg_plot_feature_widget,
    },
)

display(
    widgets.VBox([
        reg_feature_widget,
        widgets.HBox([reg_test_size_widget, reg_seed_widget]),
        widgets.HBox([reg_intercept_widget, reg_plot_feature_widget]),
        reg_output,
    ])
)


### Regression reflection questions
Answer in complete sentences:

1. Which feature set gave your **lowest MSE**?
2. Did adding more features always improve performance? Why/why not?
3. Which features appeared least helpful?
4. What does this suggest about feature selection?


# Part 2) Supervised Learning — Classification

Now we predict flower species in the Iris dataset.

### Features
- sepal length (cm)
- sepal width (cm)
- petal length (cm)
- petal width (cm)

### Your task
Use widgets to change selected features and tree depth, then compare accuracy.


In [ ]:
# Load and preview Iris data
iris = load_iris(as_frame=True)
flower_df = iris.frame.copy()
flower_df["species"] = flower_df["target"].map({
    0: "setosa",
    1: "versicolor",
    2: "virginica",
})

display(flower_df.head())
print(f"Rows: {flower_df.shape[0]}, Columns: {flower_df.shape[1]}")


## Interactive Decision Tree Lab

### Parameter guide (what each control means)
- **Features**: two numeric predictors used by the tree.
- **Class pair**: choose which two species to classify (binary so regions are red/green).
- **Max depth**: maximum number of decision levels (larger depth = more complex tree).
- **Min samples split**: minimum rows needed to split a node.
- **Test size**: fraction reserved for test evaluation.
- **Random seed**: controls reproducible train/test split.

### Exactly what to do
1. Pick exactly **2 features** for 2D decision-region plots.
2. Choose a **Class pair**.
3. Change **max depth**, **min samples split**, **test size**, and **random seed**.
4. Compare train/test accuracy and watch how the decision regions/tree change.

### Interpretation tip
- Big train accuracy but lower test accuracy can indicate overfitting.
- Simpler trees may generalize better, but too-simple trees can underfit.


In [ ]:
all_classification_features = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]

clf_feature_widget = widgets.SelectMultiple(
    options=all_classification_features,
    value=("sepal length (cm)", "petal length (cm)"),
    description="Features",
    rows=4,
    layout=widgets.Layout(width="420px"),
)

clf_pair_widget = widgets.Dropdown(
    options=[
        ("setosa vs versicolor", "setosa|versicolor"),
        ("setosa vs virginica", "setosa|virginica"),
        ("versicolor vs virginica", "versicolor|virginica"),
    ],
    value="setosa|versicolor",
    description="Class pair",
)

clf_depth_widget = widgets.IntSlider(
    value=2,
    min=1,
    max=10,
    step=1,
    description="Max depth",
    continuous_update=False,
)

clf_min_split_widget = widgets.IntSlider(
    value=2,
    min=2,
    max=20,
    step=1,
    description="Min split",
    continuous_update=False,
)

clf_test_size_widget = widgets.FloatSlider(
    value=0.2,
    min=0.1,
    max=0.5,
    step=0.05,
    description="Test size",
    continuous_update=False,
)

clf_seed_widget = widgets.IntSlider(
    value=42,
    min=0,
    max=200,
    step=1,
    description="Random seed",
    continuous_update=False,
)


def run_classifier(selected_features, class_pair, max_depth, min_samples_split, test_size, random_seed):
    if len(selected_features) != 2:
        display(Markdown("❗Please select exactly **2 features** to draw 2D decision regions."))
        return

    class_a, class_b = class_pair.split("|")
    data = flower_df[flower_df["species"].isin([class_a, class_b])].copy()

    X = data[list(selected_features)]
    y = data["species"].map({class_a: 0, class_b: 1})

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_seed, stratify=y
    )

    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_seed,
    )
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    display(Markdown(
        f"**Features:** `{list(selected_features)}` | **Classes:** `{class_a}` vs `{class_b}`"
    ))
    display(Markdown(
        f"**Max depth:** `{max_depth}` | **Min samples split:** `{min_samples_split}` | "
        f"**Test size:** `{test_size:.2f}` | **Random seed:** `{random_seed}`"
    ))
    display(Markdown(
        f"**Train accuracy:** `{train_acc:.4f}` | **Test accuracy:** `{test_acc:.4f}` (higher is better)"
    ))

    x_min, x_max = X.iloc[:, 0].min() - 0.5, X.iloc[:, 0].max() + 0.5
    y_min, y_max = X.iloc[:, 1].min() - 0.5, X.iloc[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    cmap_bg = ListedColormap(["#ffdddd", "#ddffdd"])  # red/green class regions
    cmap_pts = ListedColormap(["#d62728", "#2ca02c"])

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

    for ax, X_part, y_part, title in [
        (axes[0], X_train, y_train, "Train set regions"),
        (axes[1], X_test, y_test, "Test set regions"),
    ]:
        ax.contourf(xx, yy, Z, alpha=0.5, cmap=cmap_bg)
        ax.scatter(X_part.iloc[:, 0], X_part.iloc[:, 1], c=y_part, cmap=cmap_pts, edgecolor="k", s=35)
        ax.set_title(title)
        ax.set_xlabel(selected_features[0])
        ax.grid(alpha=0.2)

    axes[0].set_ylabel(selected_features[1])
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(11, 5))
    plot_tree(
        model,
        feature_names=list(selected_features),
        class_names=[class_a, class_b],
        filled=True,
        rounded=True,
    )
    plt.title("Decision tree structure")
    plt.show()

clf_output = widgets.interactive_output(
    run_classifier,
    {
        "selected_features": clf_feature_widget,
        "class_pair": clf_pair_widget,
        "max_depth": clf_depth_widget,
        "min_samples_split": clf_min_split_widget,
        "test_size": clf_test_size_widget,
        "random_seed": clf_seed_widget,
    },
)

display(
    widgets.VBox([
        clf_feature_widget,
        clf_pair_widget,
        widgets.HBox([clf_depth_widget, clf_min_split_widget]),
        widgets.HBox([clf_test_size_widget, clf_seed_widget]),
        clf_output,
    ])
)


### Classification reflection questions
Answer in complete sentences:

1. Which single feature gave your best accuracy?
2. How did changing `max_depth` affect performance?
3. If we had many more flower types and features, what features might matter most?
4. Would you expect the best depth to be smaller, larger, or data-dependent? Explain.


## Submission checklist
- [ ] I ran all cells in order.
- [ ] I completed at least 4 regression trials.
- [ ] I completed at least 4 classification trials.
- [ ] I answered every reflection question clearly.
